# 07 — Padrões Sazonais de Atrasos

Objetivo: identificar **quando** os atrasos são mais frequentes e intensos — por mês, dia da semana, período do dia e combinações entre eles.

Análises:
1. Sazonalidade mensal — meses críticos
2. Padrão semanal — dias da semana
3. Padrão horário — período do dia
4. Heatmaps cruzados (mês × dia, mês × período)
5. Sazonalidade por companhia aérea
6. Sazonalidade dos aeroportos anômalos (cruzamento com notebook 06)

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MONTH_NAMES = {
    1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr",
    5: "Mai", 6: "Jun", 7: "Jul", 8: "Ago",
    9: "Set", 10: "Out", 11: "Nov", 12: "Dez",
}
DOW_NAMES = {
    1: "Segunda", 2: "Terça", 3: "Quarta",
    4: "Quinta", 5: "Sexta", 6: "Sábado", 7: "Domingo",
}

## 1. Carregamento dos dados

In [ ]:
df = pl.read_parquet("../data/processed/flights_model.parquet")

# Trabalhar apenas com voos não cancelados e com delay conhecido
df = df.filter(pl.col("ARRIVAL_DELAY").is_not_null())

# Adicionar flag de atraso
df = df.with_columns([
    (pl.col("ARRIVAL_DELAY") > 0).cast(pl.Int8).alias("is_delayed"),
    (pl.col("ARRIVAL_DELAY") > 15).cast(pl.Int8).alias("is_delayed_grave"),
])

print(f"Voos: {df.height:,}")
print(f"Meses presentes: {sorted(df['MONTH'].unique().to_list())}")
df.head(3)

## 2. Sazonalidade mensal

In [ ]:
monthly = (
    df.group_by("MONTH")
    .agg([
        pl.len().alias("qtd_voos"),
        pl.col("is_delayed").mean().alias("taxa_atraso"),
        pl.col("is_delayed_grave").mean().alias("taxa_atraso_grave"),
        pl.col("ARRIVAL_DELAY").mean().alias("media_delay"),
        pl.col("ARRIVAL_DELAY").median().alias("mediana_delay"),
        pl.col("ARRIVAL_DELAY").quantile(0.90).alias("p90_delay"),
    ])
    .sort("MONTH")
    .to_pandas()
)

monthly["mes_nome"] = monthly["MONTH"].map(MONTH_NAMES)
monthly

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Taxa de atraso
ax = axes[0, 0]
bars = ax.bar(monthly["mes_nome"], monthly["taxa_atraso"],
              color=plt.cm.RdYlGn_r(monthly["taxa_atraso"] / monthly["taxa_atraso"].max()))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_title("Taxa de atraso por mês")
ax.set_xlabel("Mês")
ax.tick_params(axis="x", rotation=45)
ax.axhline(monthly["taxa_atraso"].mean(), color="gray", linestyle="--", lw=1, label="Média")
ax.legend()

# Delay médio
ax = axes[0, 1]
ax.plot(monthly["mes_nome"], monthly["media_delay"], marker="o",
        color="steelblue", lw=2, label="Média")
ax.plot(monthly["mes_nome"], monthly["mediana_delay"], marker="s",
        color="darkorange", lw=2, linestyle="--", label="Mediana")
ax.fill_between(monthly["mes_nome"], monthly["mediana_delay"],
                monthly["media_delay"], alpha=0.15, color="steelblue")
ax.set_title("Delay médio e mediana por mês (minutos)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.axhline(0, color="black", lw=0.7)

# P90 do delay
ax = axes[1, 0]
ax.bar(monthly["mes_nome"], monthly["p90_delay"],
       color="salmon", alpha=0.8)
ax.set_title("Percentil 90 do delay por mês (minutos)")
ax.set_xlabel("Mês")
ax.tick_params(axis="x", rotation=45)
ax.axhline(monthly["p90_delay"].mean(), color="gray", linestyle="--", lw=1, label="Média")
ax.legend()

# Volume de voos
ax = axes[1, 1]
ax.bar(monthly["mes_nome"], monthly["qtd_voos"] / 1000,
       color="lightsteelblue", alpha=0.9)
ax.set_title("Volume de voos por mês (mil)")
ax.set_xlabel("Mês")
ax.tick_params(axis="x", rotation=45)

plt.suptitle("Sazonalidade Mensal dos Atrasos", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Padrão semanal

In [ ]:
weekly = (
    df.group_by("DAY_OF_WEEK")
    .agg([
        pl.len().alias("qtd_voos"),
        pl.col("is_delayed").mean().alias("taxa_atraso"),
        pl.col("is_delayed_grave").mean().alias("taxa_atraso_grave"),
        pl.col("ARRIVAL_DELAY").mean().alias("media_delay"),
        pl.col("ARRIVAL_DELAY").quantile(0.90).alias("p90_delay"),
    ])
    .sort("DAY_OF_WEEK")
    .to_pandas()
)

weekly["dia_nome"] = weekly["DAY_OF_WEEK"].map(DOW_NAMES)
weekly

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

color_dow = plt.cm.RdYlGn_r(weekly["taxa_atraso"] / weekly["taxa_atraso"].max())

axes[0].bar(weekly["dia_nome"], weekly["taxa_atraso"], color=color_dow)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
axes[0].set_title("Taxa de atraso por dia da semana")
axes[0].tick_params(axis="x", rotation=40)
axes[0].axhline(weekly["taxa_atraso"].mean(), color="gray", linestyle="--", lw=1)

axes[1].bar(weekly["dia_nome"], weekly["media_delay"], color="steelblue", alpha=0.8)
axes[1].set_title("Delay médio por dia da semana (min)")
axes[1].tick_params(axis="x", rotation=40)
axes[1].axhline(0, color="black", lw=0.7)

axes[2].bar(weekly["dia_nome"], weekly["qtd_voos"] / 1000, color="lightsteelblue")
axes[2].set_title("Volume de voos por dia (mil)")
axes[2].tick_params(axis="x", rotation=40)

plt.suptitle("Padrão Semanal dos Atrasos", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Padrão por período do dia

In [ ]:
PERIODO_ORDER = ["madrugada", "manha", "tarde", "noite"]

period = (
    df.group_by("periodo_dia")
    .agg([
        pl.len().alias("qtd_voos"),
        pl.col("is_delayed").mean().alias("taxa_atraso"),
        pl.col("is_delayed_grave").mean().alias("taxa_atraso_grave"),
        pl.col("ARRIVAL_DELAY").mean().alias("media_delay"),
        pl.col("ARRIVAL_DELAY").quantile(0.90).alias("p90_delay"),
    ])
    .to_pandas()
)
period["periodo_dia"] = pd.Categorical(period["periodo_dia"], categories=PERIODO_ORDER, ordered=True)
period = period.sort_values("periodo_dia")
period

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

color_p = plt.cm.RdYlGn_r(period["taxa_atraso"].values / period["taxa_atraso"].max())

axes[0].bar(period["periodo_dia"], period["taxa_atraso"], color=color_p)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
axes[0].set_title("Taxa de atraso por período do dia")

axes[1].bar(period["periodo_dia"], period["media_delay"], color="steelblue", alpha=0.8)
axes[1].set_title("Delay médio por período (min)")
axes[1].axhline(0, color="black", lw=0.7)

axes[2].bar(period["periodo_dia"], period["qtd_voos"] / 1000, color="lightsteelblue")
axes[2].set_title("Volume de voos por período (mil)")

plt.suptitle("Padrão por Período do Dia", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Heatmaps cruzados

In [ ]:
# Mês × Dia da semana
cross_md = (
    df.group_by(["MONTH", "DAY_OF_WEEK"])
    .agg(pl.col("is_delayed").mean().alias("taxa_atraso"))
    .to_pandas()
    .pivot(index="DAY_OF_WEEK", columns="MONTH", values="taxa_atraso")
)
cross_md.index = [DOW_NAMES[i] for i in cross_md.index]
cross_md.columns = [MONTH_NAMES[c] for c in cross_md.columns]

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    cross_md,
    annot=True, fmt=".1%",
    cmap="RdYlGn_r",
    linewidths=0.4,
    ax=ax,
    cbar_kws={"format": mticker.PercentFormatter(xmax=1, decimals=0)},
)
ax.set_title("Taxa de atraso: Dia da Semana × Mês", fontsize=12)
ax.set_xlabel("Mês")
ax.set_ylabel("Dia da Semana")
plt.tight_layout()
plt.show()

In [ ]:
# Mês × Período do dia
cross_mp = (
    df.group_by(["MONTH", "periodo_dia"])
    .agg(pl.col("is_delayed").mean().alias("taxa_atraso"))
    .to_pandas()
    .pivot(index="periodo_dia", columns="MONTH", values="taxa_atraso")
)
cross_mp = cross_mp.reindex(PERIODO_ORDER)
cross_mp.columns = [MONTH_NAMES[c] for c in cross_mp.columns]

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(
    cross_mp,
    annot=True, fmt=".1%",
    cmap="RdYlGn_r",
    linewidths=0.4,
    ax=ax,
    cbar_kws={"format": mticker.PercentFormatter(xmax=1, decimals=0)},
)
ax.set_title("Taxa de atraso: Período do Dia × Mês", fontsize=12)
ax.set_xlabel("Mês")
ax.set_ylabel("Período do dia")
plt.tight_layout()
plt.show()

In [ ]:
# Dia da semana × Período do dia
cross_dp = (
    df.group_by(["DAY_OF_WEEK", "periodo_dia"])
    .agg(pl.col("is_delayed").mean().alias("taxa_atraso"))
    .to_pandas()
    .pivot(index="periodo_dia", columns="DAY_OF_WEEK", values="taxa_atraso")
)
cross_dp = cross_dp.reindex(PERIODO_ORDER)
cross_dp.columns = [DOW_NAMES[c] for c in cross_dp.columns]

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(
    cross_dp,
    annot=True, fmt=".1%",
    cmap="RdYlGn_r",
    linewidths=0.4,
    ax=ax,
    cbar_kws={"format": mticker.PercentFormatter(xmax=1, decimals=0)},
)
ax.set_title("Taxa de atraso: Período do Dia × Dia da Semana", fontsize=12)
ax.set_xlabel("Dia da Semana")
ax.set_ylabel("Período do dia")
plt.tight_layout()
plt.show()

## 6. Sazonalidade por companhia aérea

In [ ]:
# Selecionar as companhias com maior volume
top_airlines = (
    df.group_by("AIRLINE")
    .agg(pl.len().alias("qtd_voos"))
    .sort("qtd_voos", descending=True)
    .head(8)
    ["AIRLINE"]
    .to_list()
)

airline_monthly = (
    df.filter(pl.col("AIRLINE").is_in(top_airlines))
    .group_by(["AIRLINE", "MONTH"])
    .agg(pl.col("is_delayed").mean().alias("taxa_atraso"))
    .sort(["AIRLINE", "MONTH"])
    .to_pandas()
)
airline_monthly["mes_nome"] = airline_monthly["MONTH"].map(MONTH_NAMES)

# Pivot para heatmap
hm_airline = airline_monthly.pivot(index="AIRLINE", columns="MONTH", values="taxa_atraso")
hm_airline.columns = [MONTH_NAMES[c] for c in hm_airline.columns]

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    hm_airline,
    annot=True, fmt=".1%",
    cmap="RdYlGn_r",
    linewidths=0.4,
    ax=ax,
    cbar_kws={"format": mticker.PercentFormatter(xmax=1, decimals=0)},
)
ax.set_title("Taxa de atraso por Companhia × Mês", fontsize=12)
ax.set_xlabel("Mês")
ax.set_ylabel("Companhia")
plt.tight_layout()
plt.show()

In [ ]:
# Linha temporal por companhia
fig, ax = plt.subplots(figsize=(13, 5))

for airline in top_airlines:
    subset = airline_monthly[airline_monthly["AIRLINE"] == airline].sort_values("MONTH")
    ax.plot(
        subset["mes_nome"], subset["taxa_atraso"],
        marker="o", lw=2, label=airline, alpha=0.85
    )

ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_title("Evolução mensal da taxa de atraso por companhia", fontsize=12)
ax.set_xlabel("Mês")
ax.set_ylabel("Taxa de atraso")
ax.legend(title="Companhia", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Sazonalidade dos aeroportos anômalos

Cruzamento com o notebook 06: o comportamento atípico é constante ou concentrado em certos meses?

In [ ]:
# Cole aqui os aeroportos sinalizados como anômalos no notebook 06
# Exemplo: ANOMALOS = ["EWR", "JFK", "ORD"]
# Substitua pela lista real após rodar o notebook 06
ANOMALOS = []  # <- preencher

if not ANOMALOS:
    print("⚠ Lista ANOMALOS vazia. Preencha com os aeroportos do notebook 06 e execute novamente.")
else:
    anom_monthly = (
        df
        .filter(pl.col("ORIGIN_AIRPORT").is_in(ANOMALOS))
        .group_by(["ORIGIN_AIRPORT", "MONTH"])
        .agg([
            pl.col("is_delayed").mean().alias("taxa_atraso"),
            pl.col("ARRIVAL_DELAY").mean().alias("media_delay"),
        ])
        .sort(["ORIGIN_AIRPORT", "MONTH"])
        .to_pandas()
    )
    anom_monthly["mes_nome"] = anom_monthly["MONTH"].map(MONTH_NAMES)

    # Média geral (referência)
    media_geral = monthly.set_index("MONTH")["taxa_atraso"]

    fig, ax = plt.subplots(figsize=(13, 5))

    for ap in ANOMALOS:
        subset = anom_monthly[anom_monthly["ORIGIN_AIRPORT"] == ap].sort_values("MONTH")
        ax.plot(subset["mes_nome"], subset["taxa_atraso"], marker="o", lw=2, label=ap)

    ax.plot(
        [MONTH_NAMES[m] for m in sorted(media_geral.index)],
        media_geral.sort_index().values,
        color="black", lw=2, linestyle="--", label="Média geral"
    )

    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.set_title("Sazonalidade dos aeroportos anômalos vs média geral", fontsize=12)
    ax.set_xlabel("Mês")
    ax.set_ylabel("Taxa de atraso")
    ax.legend(title="Aeroporto", bbox_to_anchor=(1.01, 1), loc="upper left")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 8. Combinação crítica — pior cenário

In [ ]:
# Identificar a combinação mês + dia_semana + período com maior taxa de atraso
critical = (
    df.group_by(["MONTH", "DAY_OF_WEEK", "periodo_dia"])
    .agg([
        pl.len().alias("qtd_voos"),
        pl.col("is_delayed").mean().alias("taxa_atraso"),
        pl.col("ARRIVAL_DELAY").mean().alias("media_delay"),
    ])
    .filter(pl.col("qtd_voos") >= 100)  # mínimo para estabilidade estatística
    .sort("taxa_atraso", descending=True)
    .head(15)
    .to_pandas()
)

critical["mes_nome"] = critical["MONTH"].map(MONTH_NAMES)
critical["dia_nome"] = critical["DAY_OF_WEEK"].map(DOW_NAMES)
critical["combinacao"] = (
    critical["mes_nome"] + " / "
    + critical["dia_nome"] + " / "
    + critical["periodo_dia"]
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(critical))[::-1])
bars = ax.barh(critical["combinacao"][::-1], critical["taxa_atraso"][::-1], color=colors)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_title("Top 15 combinações com maior taxa de atraso\n(Mês / Dia da semana / Período)", fontsize=11)
ax.set_xlabel("Taxa de atraso")
ax.grid(alpha=0.2, axis="x")
plt.tight_layout()
plt.show()

print("\nTop 5 cenários críticos:")
critical[["combinacao", "taxa_atraso", "media_delay", "qtd_voos"]].head(5)

## 9. Conclusões

*(Preencha após rodar com seus dados reais)*

### Perguntas a responder com os gráficos

- **Quais meses são críticos?** Tipicamente verão (Jun–Ago) e dezembro têm os piores índices por volume de viagens e clima.
- **Qual dia da semana é mais problemático?** Sextas e domingos costumam ser os piores por efeito de acúmulo e alto volume.
- **O padrão noturno é consistente ao longo do ano?** O heatmap período × mês revela isso.
- **Alguma companhia foge ao padrão sazonal?** O gráfico de linhas por airline responde.
- **O comportamento dos aeroportos anômalos piora em meses específicos ou é constante?**

### Limitações
- Dataset de um único ano — não é possível separar tendência de sazonalidade
- `periodo_dia` é baseado no horário *programado*, não no real — partidas muito atrasadas podem mudar de período
- Feriados não estão marcados explicitamente; picos em datas comemorativas aparecem nos dados mas não são identificáveis diretamente

### Próximos passos
- Marcar feriados federais dos EUA e analisar o comportamento nas janelas de ±3 dias
- Decompor a série temporal mensal com `statsmodels` (STL decomposition) para separar tendência, sazonalidade e resíduo
- Correlacionar os meses de maior atraso com dados climáticos externos